In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mansoordaku/ckdisease")

print("Path to dataset files:", path)

100%|██████████| 9.51k/9.51k [00:00<00:00, 8.66MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/mansoordaku/ckdisease/versions/1


In [2]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. قراءة الملف المنزل
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

# 2. تنظيف بسيط وسريع للبيانات
df = df.fillna(df.mean(numeric_only=True))
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

if 'classification' in df.columns:
    df['classification'] = df['classification'].astype(str).str.strip().str.lower()
    df['classification'] = df['classification'].apply(lambda x: 1 if 'ckd' in x and 'notckd' not in x else 0)

df = pd.get_dummies(df, drop_first=True)

# 3. فصل الفحوصات (X) عن النتيجة (y) وتجهيز التدريب
X = df.drop(columns=['classification', 'id'], errors='ignore')
y = df['classification']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 4. بناء النموذج باستخدام TensorFlow
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 5. بدء التدريب
print("\nجاري تدريب النموذج عبر TensorFlow...")
model.fit(X_train, y_train, epochs=30, batch_size=10, verbose=1)

# 6. النتيجة النهائية
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\n🎉 تم تدريب النموذج بنجاح! دقة النموذج هي: {accuracy * 100:.2f}%")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



جاري تدريب النموذج عبر TensorFlow...
Epoch 1/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.5969 - loss: 0.7019
Epoch 2/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8156 - loss: 0.4614
Epoch 3/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9156 - loss: 0.3206
Epoch 4/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9375 - loss: 0.2323
Epoch 5/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9438 - loss: 0.1702
Epoch 6/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9656 - loss: 0.1292
Epoch 7/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9719 - loss: 0.0992
Epoch 8/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9812 - loss: 0.0785
Epoch 9/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9937 - loss: 0.0640
Epoch 10/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9969 - loss: 0.0525
Epoch 11/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9969 - loss: 0.0439
Epoch 12/30
32/32 ━━━━━━━━━━━━━━━━━━

In [ ]:
import gradio as gr
import numpy as np
import pandas as pd

def predict_kidney_risk(age, bp, sg, al, su, bgr, bu, sc, sod, pot, hemo, pcv, wc):
    data_dict = {col: 0 for col in X.columns}

    data_dict['age'] = age
    data_dict['bp'] = bp
    data_dict['sg'] = sg
    data_dict['al'] = al
    data_dict['su'] = su
    data_dict['bgr'] = bgr
    data_dict['bu'] = bu
    data_dict['sc'] = sc
    data_dict['sod'] = sod
    data_dict['pot'] = pot
    data_dict['hemo'] = hemo

    for col in X.columns:
        if col.startswith('pcv_') and col.endswith(str(int(pcv))):
            data_dict[col] = 1
        if col.startswith('wc_') and col.endswith(str(int(wc))):
            data_dict[col] = 1

    input_df = pd.DataFrame([data_dict])
    input_scaled = scaler.transform(input_df)

    prediction = model.predict(input_scaled, verbose=0)
    risk_score = float(prediction[0][0])

    # بناء التوصيات الصحية بناءً على الفحوصات المرتفعة
    recommendations = []

    if bp > 90:
        recommendations.append("• ضغط الدم مرتفع: ينصح بتقليل تقليل ملح الصوديوم ومتابعة القراءات بشكل دوري.")
    if sc > 1.2 or bu > 40:
        recommendations.append("• مؤشرات وظائف الكلى (الكرياتينين/اليوريا) مرتفعة: ينصح بمراجعة طبيب أخصائي إجراء فحوصات دقيقة.")
    if al > 0:
        recommendations.append("• وجود زلال (ألبومين) في البول: يستوجب المتابعة للحد من الإجهاد الكلوي.")
    if bgr > 140 or su > 0:
        recommendations.append("• مستويات السكر مرتفعة: يوصى بضبط مستوى الجلوكوز في الدم لحماية الشعيرات الدموية للكلى.")
    if hemo < 11:
        recommendations.append("• انخفاض الهيموجلوبين (فقر دم): يتطلب تقييم طبي للتأكد من استقرار نسبة الحديد وإفراز نظام الإريثروبويتين.")

    # صياغة النتيجة النهائية
    if risk_score > 0.5:
        res = f"⚠️ مستوى الخطر: مرتفع جداً ({risk_score*100:.1f}%)\n"
        res += "--------------------------------------------------\n"
        res += "💡 التوصيات والإرشادات السريرية التوعوية:\n"
        if recommendations:
            res += "\n".join(recommendations)
        else:
            res += "• يُنصح بمراجعة طبيب أخصائي كلى لإجراء الفحوصات التفصيلية."
        return res
    else:
        res = f"✅ مستوى الخطر: منخفض ({risk_score*100:.1f}%)\n"
        res += "--------------------------------------------------\n"
        res += "💡 التوصيات العامة:\n"
        res += "• النتائج الأولية ضمن النطاق الطبيعي. حافظ على نمط حياة صحي وشرب كميات كافية من الماء يومياً."
        return res

inputs = [
    gr.Slider(minimum=1, maximum=100, value=40, step=1, label="العمر (age)"),
    gr.Slider(minimum=50, maximum=180, value=80, step=5, label="ضغط الدم (bp)"),
    gr.Slider(minimum=1.005, maximum=1.025, value=1.020, step=0.005, label="الكثافة النوعية (sg) - عشري"),
    gr.Slider(minimum=0, maximum=5, value=0, step=1, label="الألبومين (al)"),
    gr.Slider(minimum=0, maximum=5, value=0, step=1, label="السكر بالبول (su)"),
    gr.Slider(minimum=50, maximum=500, value=120, step=1, label="سكر الدم (bgr)"),
    gr.Slider(minimum=10, maximum=200, value=30, step=1, label="اليوريا (bu)"),
    gr.Slider(minimum=0.4, maximum=15.0, value=1.0, step=0.1, label="الكرياتينين (sc) - عشري"),
    gr.Slider(minimum=100, maximum=160, value=138, step=1, label="الصوديوم (sod)"),
    gr.Slider(minimum=2.5, maximum=8.0, value=4.2, step=0.1, label="البوتاسيوم (pot) - عشري"),
    gr.Slider(minimum=3.0, maximum=18.0, value=15.0, step=0.1, label="الهيموجلوبين (hemo) - عشري"),
    gr.Slider(minimum=15, maximum=55, value=44, step=1, label="مكداس الدم (pcv)"),
    gr.Slider(minimum=2200, maximum=26000, value=8400, step=100, label="كريات الدم البيضاء (wc)")
]

demo = gr.Interface(
    fn=predict_kidney_risk,
    inputs=inputs,
    outputs="text",
    title="🏥 نظام التنبؤ المبكر بخطر الإصابة بمرض الكلى",
    description="أداة ذكاء اصطناعي مساعدة للتنبؤ الأولي والتثقيف الصحي برعاية الصحة العامة."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9334ce411095bdd2b0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
